# 02 · Evaluate the base model

This notebook measures the **base model** (Gemma 4 E2B, before any of our training) on the puzzle sets from `01_data_pipeline.ipynb`. Its scores are the baseline that every trained model is compared against.

For each puzzle, the model gets the position as a prompt and answers with a move. We then grade the answer:
- **puzzle accuracy:** does the move match the solution? (In mate-in-1 puzzles, any checkmating move counts.)
- **legal-move rate:** is the answer a legal move? Failures are split into *illegal* and *unparseable*.
- **ACPL:** for legal moves, how much worse the move is than Stockfish's best move, in centipawns

Decoding is greedy (the model's single most likely answer), so results are reproducible.

**Needs an NVIDIA GPU.** The first run downloads the model (about 10 GB).

**Outputs:** `results/base_dev.json`, `results/base.json`, and one row each in `results/summary.csv`.

In [1]:
# Settings
RUN_NAME = "base"
PUZZLE_LIMIT = None   # None = all puzzles. Set a small number (e.g. 5) for a quick check.

import json

import pandas as pd
import torch

from chesslm.config import REPO_ROOT, load_config
from chesslm.engine import MoveScorer
from chesslm.evaluate import evaluate, load_eval_model, save_run, summarize

model_cfg = load_config("sft")["model"]
prompt_cfg = load_config("prompt")
eval_cfg = load_config("eval")


def run_eval(model, tokenizer, split, run_name, adapter=None):
    """Grade the model on one puzzle set, save the results, and return the summary."""
    puzzles = pd.read_csv(REPO_ROOT / "data" / "eval" / f"puzzles_{split}.csv")
    if PUZZLE_LIMIT:
        puzzles = puzzles.head(PUZZLE_LIMIT)
    with MoveScorer(eval_cfg["stockfish"]) as scorer:
        graded = evaluate(model, tokenizer, puzzles, prompt_cfg, eval_cfg["generation"], scorer)
    summary = summarize(graded)
    config = {"model": model_cfg, "adapter": adapter, "prompt": prompt_cfg, "eval": eval_cfg, "split": split}
    save_run(run_name, graded, summary, config)
    print(f"{run_name}: {len(puzzles)} puzzles | accuracy {summary['puzzle_accuracy']:.1%} | "
          f"legal {summary['legal_rate']:.1%} | ACPL {summary['acpl'] if summary['acpl'] is not None else float('nan'):.0f} | "
          f"{summary['seconds_per_position']:.2f} s per puzzle")
    return graded, summary

## 1. Load the model

We load only the language part of Gemma. The checkpoint also contains image and audio encoders, which we don't use.

In [2]:
tokenizer, model = load_eval_model(model_cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model_cfg['id']} on {model.device}")
print(f"Parameters (language model only): {n_params / 1e9:.2f} billion")

Loading weights:   0%|          | 0/540 [00:00<?, ?it/s]

Model: google/gemma-4-E2B-it on cuda:0
Parameters (language model only): 4.63 billion


## 2. A first look

Three dev puzzles: the correct answer, and what the model actually replies.

In [3]:
from chesslm.evaluate import generate_replies
from chesslm.prompts import build_prompt

examples = pd.read_csv(REPO_ROOT / "data" / "eval" / "puzzles_dev.csv").head(3)
prompts = [build_prompt(tokenizer, row, prompt_cfg) for row in examples.to_dict("records")]
replies, _ = generate_replies(model, tokenizer, prompts, eval_cfg["generation"])
for (_, row), reply in zip(examples.iterrows(), replies):
    print(f"Puzzle {row.puzzle_id} (rating {row.rating}): solution {row.answer_san} ({row.answer_uci}) | model replied {reply!r}")

Puzzle 551hv (rating 1659): solution Qxe6+ (e4e6) | model replied 'Qd3'
Puzzle UKTzv (rating 1361): solution Qh4 (d8h4) | model replied '...Nf6'
Puzzle R0wwC (rating 1209): solution Nxf3 (d4f3) | model replied 'Rc1d3'


## 3. Dev set (200 puzzles)

A first full pass, on the development set.

In [4]:
dev_graded, dev_summary = run_eval(model, tokenizer, "dev", f"{RUN_NAME}_dev")

base_dev: 200 puzzles | accuracy 1.5% | legal 44.0% | ACPL 847 | 0.24 s per puzzle


## 4. Test set (1,000 puzzles)

The main measurement. These scores go in the report.

In [5]:
test_graded, test_summary = run_eval(model, tokenizer, "test", RUN_NAME)

print("\nAccuracy by rating band:")
for band, accuracy in test_summary["accuracy_by_band"].items():
    print(f"  {band:>10}: {accuracy:.1%}")
print(f"\nOutcomes: legal {test_summary['legal_rate']:.1%}, illegal {test_summary['illegal_rate']:.1%}, "
      f"unparseable {test_summary['unparseable_rate']:.1%}")

base: 1000 puzzles | accuracy 2.8% | legal 41.2% | ACPL 828 | 0.22 s per puzzle

Accuracy by rating band:
   1200-1800: 4.8%
   1800-2400: 2.4%
       2400+: 1.6%
       <1200: 2.4%

Outcomes: legal 41.2%, illegal 45.8%, unparseable 13.0%


A few replies of each kind, to see what the model actually writes:

In [6]:
for outcome, group in test_graded.groupby("outcome"):
    print(f"--- {outcome} ({len(group)} replies)")
    for row in group.head(3).itertuples():
        print(f"    {row.reply!r}")

--- illegal (458 replies)
    'c2c3'
    '...d2d5'
    'Rde1'
--- legal (412 replies)
    'Qf7'
    '...Qc7'
    'Qd3'
--- unparseable (130 replies)
    'R7R/1R5R/1p2rp2/p7/8/1p5P/5PP1/6K1'
    'Rfc1rk1/p1p2pp1/3p3p/7n/P1pBP1b1/2P3'
    'rh4g3'


## Summary

The base model's scores are saved in `results/summary.csv` (runs `base_dev` and `base`). Next, `03_sft_first_run.ipynb` trains the first model and grades it in exactly the same way.